# Análise Exploratória de Dados (EDA)
``Análise e Normalização Relacional de Contratos e Acordos de Telecomunicações (Anatel)``

---

**Autores**: **Bruna Gomes** - 2023061070,

**Carlos Eduardo Oliveira** - 2021020058,

**Emanuel Figueiredo** - 2019092470, 

**Gabriel Veloso** - 2024421649, 

**Victoria Reis** - 2020104630.

---

Este notebook tem como objetivo realizar uma Análise Exploratória de Dados (EDA) do conjunto de dados "**Competição – Empresas Credenciadas de Rede Virtual**", disponibilizado pela Anatel. A análise busca compreender as características da base de dados, identificar padrões, inconsistências e anomalias, bem como gerar insights que auxiliem no processo de modelagem e normalização relacional dos dados.

``[https://dados.gov.br/dados/conjuntos-dados/empresas-credenciadas-de-rede-virtual-do-servico-movel-pessoal-telefonia-movel]``

## Bibliotecas Utilizadas

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re

## Tratamento dos dados

### Funções utilizadas no tratamento dos dados

In [2]:
# limpeza de caracteres
def limpar_texto(texto):
    if pd.isna(texto):
        return texto
    
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(
        c for c in texto
        if unicodedata.category(c) != 'Mn'
    )

    texto = re.sub(r'[^a-z0-9\s]', '', texto)

    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

# limpeza especifica para casos de informe sei
def limpeza_sei(texto):
    if pd.isna(texto):
        return texto
    
    texto = str(texto).lower()
    
    if texto.find('/') > -1: 
        texto = texto[:texto.find('/')]
    if texto.find(' ') > -1:
        texto = texto[:texto.find(' ')]
    texto = re.sub(r'[a-zA-Z///*]', '', texto).strip()
    if not texto: 
        return '0'
    else: 
        return texto

# aplicacao de limpeza de caracteres no df
def tratar_strings_df(df, exceptions:list[str]=[]):
    df = df.copy()

    colunas_texto = df.select_dtypes(include=['object', 'string']).columns

    for coluna in colunas_texto:
        if coluna not in exceptions:
            df[coluna] = df[coluna].apply(limpar_texto)

    return df

# limpeza de coluna de datas
def coluna_data(df, list_columns):
    for col in df.columns:
        if col in list_columns:
            df[col] = pd.to_datetime(
                df[col],
                format='%d/%m/%Y',
                errors='coerce'
            )

    return df

# limpeza para colunas de cnpj
def tratar_cnpj(df, cnpj_colunas):
    for col in df.columns:
        if col in cnpj_colunas:
            df[col] = (df[col]
                       .fillna(0)
                       .astype(float)
                       .astype(int)
                       .astype(str)
                       .str.zfill(14))
            
    return df

## **EDA** 

O conjunto de dados analisado é composto por diferentes bases, em que cada uma representa uma versão específica de contratos ou acordos regulados pela Agência Nacional de Telecomunicações (Anatel). Esses contratos são agrupados em quatro categorias principais: **Interconexão**, **Comunicação**, **MVNO (Mobile Virtual Network Operator)** e **RAN Sharing**. Cada categoria possui características próprias, refletindo diferentes formas de relacionamento e compartilhamento de recursos entre as empresas do setor de telecomunicações.

Nesta análise, utilizaremos a linguagem Python para explorar individualmente cada conjunto de dados, buscando compreender sua estrutura, identificar padrões, verificar a qualidade das informações e levantar aspectos relevantes para a modelagem e normalização relacional dos dados.

> As categorias de contratos podem ser resumidas da seguinte forma:
>
> * **Interconexão:** contratos que estabelecem a interligação entre redes de diferentes prestadoras de telecomunicações, permitindo que usuários de uma rede possam se comunicar com usuários de outra rede.
>
> * **Comunicação:** acordos relacionados ao compartilhamento ou fornecimento de serviços de comunicação entre operadoras, abrangendo aspectos operacionais e comerciais necessários para a prestação dos serviços.
>
> * **MVNO (Mobile Virtual Network Operator):** contratos que permitem que uma empresa atue como operadora móvel virtual, oferecendo serviços de telefonia móvel aos seus clientes sem possuir infraestrutura própria de rede, utilizando a rede de uma operadora hospedeira.
>
> * **RAN Sharing (Radio Access Network Sharing):** acordos de compartilhamento da infraestrutura de acesso rádio entre operadoras, possibilitando o uso conjunto de antenas, estações rádio-base e outros elementos da rede móvel, com o objetivo de reduzir custos e ampliar a cobertura dos serviços.
>

### **Interconexões** 

In [3]:
# contratos interconexao
df_contratos_int = pd.read_csv(
    r'Dados - Contratos/contratos_interconexao.csv',
    sep=';')

print(df_contratos_int.shape)
df_contratos_int.head(3)

(12374, 15)


,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO
0,53500.028940/2006-92,26/10/2006,NÃO INFORMADO,2449992000164,VIVO S.A.,10,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,20,NaN,-1,06/10/2014,06/10/2014,Classe: IV. Localização: ARQUIVO GERAL.
1,53500.004342/2007-17,14/02/2007,CONTRATO,2449992000164,VIVO S.A.,10,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,20,NaN,47,05/08/2015,19/08/2015,Classe: IV. Localização: ESTAÇÃO.
2,53500.021964/2008-82,14/08/2008,NÃO INFORMADO,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,10,NaN,5958690000100,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,NaN,3,13/06/2013,17/06/2013,Classe: IV. Localização: ARQUIVO SETORIAL.


In [4]:
df_contratos_int['ACORDO_TIPO'].unique()

array(['NÃO INFORMADO', 'CONTRATO', 'ADITIVO'], dtype=object)

In [5]:
df_contratos_int = df_contratos_int.drop_duplicates()
df_contratos_int.shape

(12364, 15)

In [6]:
df_contratos_int['PROCESSO_ANATEL'].unique()

array(['53500.028940/2006-92', '53500.004342/2007-17',
       '53500.021964/2008-82', ..., '53500.057660/2024-73',
       '53500.060181/2024-34', '53500.019526/2025-55'],
      shape=(1735,), dtype=object)

In [7]:
dup_int = df_contratos_int['PROCESSO_ANATEL'][df_contratos_int['PROCESSO_ANATEL'].duplicated()].unique()

print(dup_int)

['53500.028794/2011-62' '53500.016955/2007-99' '53500.005631/2012-92' ...
 '53500.057660/2024-73' '53500.060181/2024-34' '53500.019526/2025-55']


In [8]:
df_contratos_int[df_contratos_int['PROCESSO_ANATEL'] == '53500.028794/2011-62'].head(3)

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO
9,53500.028794/2011-62,21/12/2011,CONTRATO,76535764000143,Oi S.a. - em Recuperacao Judicial,171,LDI,8868001000164,HOJE SISTEMAS DE INFORMATICA -EM RECUPERAÇÃO J...,171,LDI,5252,30/10/2013,01/11/2013,Classe: I-770.
10,53500.028794/2011-62,21/12/2011,CONTRATO,76535764000143,Oi S.a. - em Recuperacao Judicial,171,LDI,8868001000164,HOJE SISTEMAS DE INFORMATICA -EM RECUPERAÇÃO J...,171,LDN,5252,30/10/2013,01/11/2013,Classe: I-770.
11,53500.028794/2011-62,21/12/2011,CONTRATO,76535764000143,Oi S.a. - em Recuperacao Judicial,171,LDI,8868001000164,HOJE SISTEMAS DE INFORMATICA -EM RECUPERAÇÃO J...,171,LOCAL,5252,30/10/2013,01/11/2013,Classe: I-770.


In [9]:
df_contratos_int[df_contratos_int['PROCESSO_ANATEL'] == '53500.016955/2007-99'].head(3)

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO
18,53500.016955/2007-99,09/01/2012,CONTRATO,4533132000130,CONECTA TELECOMUNICACOES S.A.,171,LDI,2558157000162,Telefonica Brasil S.a.,171,LDI,2613,19/04/2013,17/05/2013,Classe: I-383. Localização: ARQUIVO SETORIAL.
19,53500.016955/2007-99,09/01/2012,CONTRATO,4533132000130,CONECTA TELECOMUNICACOES S.A.,171,LDI,2558157000162,Telefonica Brasil S.a.,171,LDN,2613,19/04/2013,17/05/2013,Classe: I-383. Localização: ARQUIVO SETORIAL.
20,53500.016955/2007-99,09/01/2012,CONTRATO,4533132000130,CONECTA TELECOMUNICACOES S.A.,171,LDI,2558157000162,Telefonica Brasil S.a.,171,LOCAL,2613,19/04/2013,17/05/2013,Classe: I-383. Localização: ARQUIVO SETORIAL.


In [10]:
print(df_contratos_int['MODALIDADE_STFC_1'].unique())
print(df_contratos_int['MODALIDADE_STFC_2'].unique())
print(df_contratos_int['SERVICO_1'].unique())
print(df_contratos_int['SERVICO_2'].unique())
print(df_contratos_int[['SERVICO_1', 'MODALIDADE_STFC_1']].value_counts())
print(df_contratos_int[['SERVICO_2', 'MODALIDADE_STFC_2']].value_counts())

[nan 'LDI' 'LDN' 'LOCAL']
[nan 'LDI' 'LDN' 'LOCAL']
[ 10  20 171  45]
[ 20  10 171  45]
SERVICO_1  MODALIDADE_STFC_1
171        LOCAL                3218
           LDN                  3107
           LDI                  3086
Name: count, dtype: int64
SERVICO_2  MODALIDADE_STFC_2
171        LOCAL                4891
           LDN                  3689
           LDI                  3661
Name: count, dtype: int64


In [12]:
df_contratos_int[df_contratos_int['SERVICO_1'] == 10]

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO
0,53500.028940/2006-92,26/10/2006,NÃO INFORMADO,2449992000164,VIVO S.A.,10,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,20,NaN,-1,06/10/2014,06/10/2014,Classe: IV. Localização: ARQUIVO GERAL.
1,53500.004342/2007-17,14/02/2007,CONTRATO,2449992000164,VIVO S.A.,10,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,20,NaN,47,05/08/2015,19/08/2015,Classe: IV. Localização: ESTAÇÃO.
2,53500.021964/2008-82,14/08/2008,NÃO INFORMADO,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,10,NaN,5958690000100,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,NaN,3,13/06/2013,17/06/2013,Classe: IV. Localização: ARQUIVO SETORIAL.
3,53500.004720/2009-16,04/03/2009,NÃO INFORMADO,2449992000164,VIVO S.A.,10,NaN,5958690000100,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,NaN,2,13/06/2013,17/06/2013,Classe: IV. Localização: ARQUIVO SETORIAL.
5,53500.009336/2009-18,16/04/2009,NÃO INFORMADO,4206050000180,TIM CELULAR S.A.,10,NaN,5958690000100,UNICEL DO BRASIL TELECOMUNICAÇÕES LTDA.,10,NaN,1,13/06/2013,17/06/2013,Classe: IV. Localização: ARQUIVO SETORIAL.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12351,53500.019526/2025-55,17/03/2025,CONTRATO,2421421000111,TIM S A,10,NaN,15174013000154,Servstar Comercio e Servicos em Telefonia e In...,171,LDN,13578256,17/04/2025,17/04/2025,Homologado por ORPA/OPI
12352,53500.019526/2025-55,17/03/2025,CONTRATO,2421421000111,TIM S A,10,NaN,15174013000154,Servstar Comercio e Servicos em Telefonia e In...,171,LOCAL,13578256,17/04/2025,17/04/2025,Homologado por ORPA/OPI
12362,53500.019526/2025-55,17/03/2025,CONTRATO,2421421000111,TIM S A,10,NaN,28135347000160,4NET SOLUCOES EM TELECOMUNICACOES LTDA,171,LDI,13578256,17/04/2025,17/04/2025,Homologado por ORPA/OPI
12363,53500.019526/2025-55,17/03/2025,CONTRATO,2421421000111,TIM S A,10,NaN,28135347000160,4NET SOLUCOES EM TELECOMUNICACOES LTDA,171,LDN,13578256,17/04/2025,17/04/2025,Homologado por ORPA/OPI


In [13]:
df_contratos_int.dtypes

PROCESSO_ANATEL            object
PROTOCOLO_DATA             object
ACORDO_TIPO                object
PRESTADORA_1_CNPJ           int64
PRESTADORA_1               object
SERVICO_1                   int64
MODALIDADE_STFC_1          object
PRESTADORA_2_CNPJ           int64
PRESTADORA_2               object
SERVICO_2                   int64
MODALIDADE_STFC_2          object
DESPACHO_DECISORIO_SEI     object
DESPACHO_DECISORIO_DATA    object
CONCLUSAO_DATA             object
OBSERVACAO                 object
dtype: object

In [14]:
df_contratos_int.isna().all()

PROCESSO_ANATEL            False
PROTOCOLO_DATA             False
ACORDO_TIPO                False
PRESTADORA_1_CNPJ          False
PRESTADORA_1               False
SERVICO_1                  False
MODALIDADE_STFC_1          False
PRESTADORA_2_CNPJ          False
PRESTADORA_2               False
SERVICO_2                  False
MODALIDADE_STFC_2          False
DESPACHO_DECISORIO_SEI     False
DESPACHO_DECISORIO_DATA    False
CONCLUSAO_DATA             False
OBSERVACAO                 False
dtype: bool

In [15]:
df_contratos_int.duplicated().any()

np.False_

In [16]:
df_contratos_int.DESPACHO_DECISORIO_SEI.value_counts().sort_index()

DESPACHO_DECISORIO_SEI
-1             43
1               2
1/2014/SEI      2
1/2015/SEI      2
10             12
             ... 
9812111         9
9832857        12
9832888       120
99              5
DSP ORD         3
Name: count, Length: 1645, dtype: int64

In [17]:
print(df_contratos_int.PRESTADORA_1.isna().value_counts())
print(df_contratos_int.PRESTADORA_2.isna().value_counts())

PRESTADORA_1
False    12364
Name: count, dtype: int64
PRESTADORA_2
False    12364
Name: count, dtype: int64


In [18]:
df_contratos_int[['SERVICO_1', 'MODALIDADE_STFC_1']].drop_duplicates()

,SERVICO_1,MODALIDADE_STFC_1
0,10,NaN
4,20,NaN
9,171,LDI
12,171,LDN
15,171,LOCAL
2431,45,NaN


In [19]:
df_contratos_int[['SERVICO_2', 'MODALIDADE_STFC_2']].drop_duplicates()

,SERVICO_2,MODALIDADE_STFC_2
0,20,NaN
2,10,NaN
9,171,LDI
10,171,LDN
11,171,LOCAL
2431,45,NaN


### **Compartilhamento**

In [21]:
# contratos de compartilhamentos
df_contratos_comp = pd.read_csv(
    r'Dados - Contratos/contratos_compartilhamento.csv',
    sep = ';'
)

print(df_contratos_comp.shape)
df_contratos_comp.head(5)

(9630, 11)


,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,DETENTORA,DETENTORA_CNPJ,SOLICITANTE,SOLICITANTE_CNPJ,INFORME_SEI,INFORME_DATA,CONCLUSAO_DATA,OBSERVACAO
0,CONTRATO,22/04/2013,53500.009204/2013-64,LIGHT SERVIÇOS DE ELETRICIDADE S.A,6.044444e+13,AFINET,NaN,70/2013-CPRP/SCP,17/06/2013,17/06/2013,Processo ANEEL: 48500.002273/2013-15. Despacho...
1,CONTRATO,02/05/2013,53500.011549/2013-88,CAIUÁ - DISTRIBUIÇÃO DE ENERGIA S.A.,7.282377e+12,ACER TELECOMUNICAÇÕES,NaN,66/2013-CPRP/SCP,14/06/2013,17/06/2013,Processo ANEEL: 48500.001859/2013-54. \r
2,CONTRATO,06/05/2013,53500.011548/2013-33,CAIUÁ - DISTRIBUIÇÃO DE ENERGIA S.A.,7.282377e+12,NOVA PORTONET,NaN,120/2013-CPRP/SCP,28/06/2013,03/07/2013,Processo ANEEL: 48500.001858/2013-18. Despacho...
3,CONTRATO,06/05/2013,53500.012976/2013-83,COPEL DISTRIBUIÇÃO S.A.,4.368898e+12,CIABRASNET,NaN,73/2013-CPRP/SCP,18/06/2013,18/06/2013,Processo ANEEL: 48500.002187/2013-02. \r
4,CONTRATO,06/05/2013,53500.012976/2013-83,COPEL DISTRIBUIÇÃO S.A.,4.368898e+12,DELTA TELECOMUNICAÇÕES,NaN,73/2013-CPRP/SCP,18/06/2013,18/06/2013,Processo ANEEL: 48500.002271/2013-18. Despacho...


In [22]:
df_contratos_comp['INFORME_SEI'].duplicated()

0       False
1       False
2       False
3       False
4        True
        ...  
9625     True
9626     True
9627     True
9628     True
9629     True
Name: INFORME_SEI, Length: 9630, dtype: bool

In [23]:
df_contratos_comp.loc[9629]

ACORDO_TIPO                      CONTRATO
PROTOCOLO_DATA                 29/04/2025
PROCESSO_ANATEL      53500.031783/2025-65
DETENTORA           AMAZONAS ENERGIA S.A.
DETENTORA_CNPJ            2341467000120.0
SOLICITANTE          T. S. CASTELO BRANCO
SOLICITANTE_CNPJ         58095638000133.0
INFORME_SEI                      13722855
INFORME_DATA                   27/05/2025
CONCLUSAO_DATA                 30/05/2025
OBSERVACAO                             \r
Name: 9629, dtype: object

In [24]:
df_contratos_comp.loc[9628]

ACORDO_TIPO                                        CONTRATO
PROTOCOLO_DATA                                   29/04/2025
PROCESSO_ANATEL                        53500.031783/2025-65
DETENTORA                             AMAZONAS ENERGIA S.A.
DETENTORA_CNPJ                              2341467000120.0
SOLICITANTE         T G L SERVIÇOS DE TELECOMUNICAÇÕES LTDA
SOLICITANTE_CNPJ                           28450243000140.0
INFORME_SEI                                        13722855
INFORME_DATA                                     27/05/2025
CONCLUSAO_DATA                                   30/05/2025
OBSERVACAO                                               \r
Name: 9628, dtype: object

In [25]:
df_contratos_comp.dtypes

ACORDO_TIPO          object
PROTOCOLO_DATA       object
PROCESSO_ANATEL      object
DETENTORA            object
DETENTORA_CNPJ      float64
SOLICITANTE          object
SOLICITANTE_CNPJ    float64
INFORME_SEI          object
INFORME_DATA         object
CONCLUSAO_DATA       object
OBSERVACAO           object
dtype: object

In [26]:
df_contratos_comp.isna().all()

ACORDO_TIPO         False
PROTOCOLO_DATA      False
PROCESSO_ANATEL     False
DETENTORA           False
DETENTORA_CNPJ      False
SOLICITANTE         False
SOLICITANTE_CNPJ    False
INFORME_SEI         False
INFORME_DATA        False
CONCLUSAO_DATA      False
OBSERVACAO          False
dtype: bool

In [27]:
df_contratos_comp.duplicated().any()

np.True_

In [28]:
df_contratos_comp.ACORDO_TIPO.unique()

array(['CONTRATO', 'TERMO ADITIVO 2', 'CORRESPONDÊNCIA DA DETENTORA',
       'TERMO ADITIVO 1', 'TERMO ADITIVO 2 E 3', 'TERMOS ADITIVOS 3 A 6',
       'TERMO DE COMPROMISSO PARA COMPARTILHAMENTO', 'TERMO ADITIVO 3',
       'TERMOS ADITIVOS 1 E 2', 'ADITIVO', 'CONTRATO E ADITIVO',
       'CONTRATO E ADITIVOS 1 E 2', 'TERMO DE CESSÃO', 'TERMO ADITIVO',
       '1º E 2º TERMO ADITIVO', '3º TERMO ADITIVO', 'CONTRATO (SCM)',
       '1º TERMO ADITIVO', 'ADITIVO A TERMO DE COMPROMISSO',
       'TERMO DE COOPERAÇÃO TÉCNICA', 'CARTA DA DISTRIBUIDORA',
       '4º TERMO ADITIVO', '2º TERMO ADITIVO', 'TERMO ADITIVO 4',
       '5º TERMO ADITIVO', '6º TERMO ADITIVO', '7º TERMO ADITIVO',
       '8º TERMO ADITIVO', '9º TERMO ADITIVO', '10º TERMO ADITIVO',
       '11º TERMO ADITIVO', 'OFÍCIO DA ANEEL', 'TERMO ADITVO 4',
       'TERMO ADITIVO 5', 'TERMO ADITIVO 6', 'TERMO ADITIVO 7',
       'TERMO ADITVO 8', 'TERMO ADITIVO 9', 'TERMO ADITIVO 10',
       'TERMO ADITIVO 11', 'TERMO ADITIVO 12', 'TERMO ADIT

In [29]:
df_contratos_comp.DETENTORA.isna().value_counts()

DETENTORA
False    9630
Name: count, dtype: int64

In [30]:
df_contratos_comp.DETENTORA.sample(20)

3668    ELETROPAULO METROPOLITANA ELETRICIDADE DE SÃO ...
5021    ELETROPAULO METROPOLITANA ELETRICIDADE DE SÃO ...
6415          COMPANHIA PIRATININGA DE FORÇA E LUZ - CPFL
7528                RGE SUL DISTRIBUIDORA DE ENERGIA S.A.
1477                             CELESC DISTRIBUIÇÃO S.A.
2867               COMPANHIA ENERGÉTICA DO CEARÁ - COELCE
8892                              CEMIG DISTRIBUIÇÃO S.A.
4479                        AMPLA ENERGIA E SERVIÇOS S.A.
5110               COMPANHIA ENERGÉTICA DO CEARÁ - COELCE
2379             COMPANHIA PAULISTA DE FORÇA E LUZ - CPFL
5312    COOPERATIVA DE ELETRIFICAÇÃO RURAL DE ITAÍ-PAR...
3103                        AMPLA ENERGIA E SERVIÇOS S.A.
3730    COMPANHIA ESTADUAL DE DISTRIBUIÇÃO DE ENERGIA ...
3873                HIDROPAN DISTRIBUIÇÃO DE ENERGIA S.A.
1249    COMPANHIA ENERGÉTICA DO RIO GRANDE DO NORTE S/...
5873                              CEMIG DISTRIBUIÇÃO S.A.
2750                         RIO GRANDE ENERGIA S.A - RGE
4535          

In [31]:
df_contratos_comp.SOLICITANTE.isna().value_counts()

SOLICITANTE
False    9630
Name: count, dtype: int64

In [32]:
df_contratos_comp.SOLICITANTE.sample(20)

4771             TELNET SISTEMAS E COMUNICAÇÕES EIRELI ME
1291                   PRONET PROVEDOR NETWORK WIFI LTDA.
1456                  GRUPO G1 TELECOMUNICAÇÕES LTDA - ME
1686         SANTOS OLIVEIRE SERVIÇOS DE MULTIMÍDIA LTDA.
3767                                LUQUINI E BAIOCO LTDA
9507                                   IPNET TELECOM LTDA
8706                         MANAOS TELECOMUNICAÇÕES LTDA
9413           LORD SERVIÇO DE INTERNET FIBRA ÓPTICA LTDA
3902                              ARANET TELECOM CIA LTDA
53                               REDE POPULAR DE INTERNET
1937                              LAZERNET.COM.BR LTDA ME
7050    TESSI - SERVICOS DE TELECOMUNICACOES LTDA (TES...
5835    SULNET RC INTERNET PROVIDER INFOPRMÁTICA LTDA ...
7086                            ANTONIO CARLOS DOS SANTOS
1246    NOVA INFRAESTRUTURA - LOCAÇÕES E PARTICIPAÇÕES...
4022    EMPRESA BRASILEIRA DE TELECOMUNICAÇÕES S.A. - ...
8199                             TILOG TELECOM LTDA (TI5)
3188          

In [33]:
df_contratos_comp.INFORME_SEI.sample(20)

3898          4303305
5282          6676034
3036              NaN
483     Não aplicável
1668           744577
8858         12158913
7659          9416362
5791          7453897
8248         11170419
1171           373727
3497          3231794
2383          1568178
8989         12298105
7609          9416362
7383          9382426
4398          5386862
1296           176403
9062         12432941
7091          9042895
3886          4381799
Name: INFORME_SEI, dtype: object

In [34]:
def normalizar_texto(texto:str)->str:
    if pd.isna(texto):
        return texto
    novo_texto = str(texto).strip()
    novo_texto = re.sub(r'\s+', ' ', novo_texto)
    return novo_texto

new_cols = (
    df_contratos_comp['INFORME_SEI']
    .apply(normalizar_texto)
    .str.upper()
)

new_cols[(~new_cols.isna()) & (new_cols.map(lambda x: str(x)[0].isalpha()))].value_counts()

INFORME_SEI
NÃO APLICÁVEL          73
COM PENDÊNCIA          60
DESNECESSÁRIO          32
NÃO INFORMADO          16
NÃO SE APLICA          13
PRESTAÇÃO IRREGULAR     3
NÃO HOUVE               1
ARQUIVO NÃO ABRE        1
Name: count, dtype: int64

### MVNO 

In [36]:
# contratos mvno (mobile virtual network operator)
df_contratos_mvno = pd.read_csv(
    r'Dados - Contratos/contratos_mvno.csv',
    sep = ';'
)

print(df_contratos_mvno.shape)
df_contratos_mvno.head(3)

(341, 13)


,ACORDO_TIPO,PROTOCOLO_DATA,PROCESSO_ANATEL,PRESTADORA_ORIGEM,PRESTADORA_ORIGEM_CNPJ,CREDENCIADA,CREDENCIADA_CNPJ,VIGENCIA_DATA_FIM,PROCESSO_DESCREDENCIAMENTO,OBSERVACAO,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA
0,CONTRATO,07/01/2015,53500.000262/2015-94,Telefonica Brasil S.a.,2558157000162,Mais AD Credenciada de Telefonia S.A.,17751901000118,07/04/2020,53500.000262/2015-94,Publicação no DOU.,201590015890,30/01/2015,30/01/2015
1,CONTRATO,14/03/2016,53504.001925/2016-39,Datora Mobile Telecomunicacoes S.a.,18384930000151,BT Brasil Serviços de Telecomunicações Ltda. (...,33179565000137,VIGENTE,NaN,Não identificado,588202,20/07/2016,20/07/2016
2,CONTRATO,23/03/2016,53500.004886/2016-61,SURF TELECOM SA,10455746000143,Always Tecnologia Ltda. (Igreja da Fé),69190577000100,21/07/2017,NaN,Não identificado,591183,20/07/2016,20/07/2016


In [37]:
unicos = df_contratos_mvno['CREDENCIADA'].unique()
unicos.size

227

In [38]:
u = df_contratos_mvno['PROCESSO_ANATEL'].unique()
u.size

245

In [39]:
df_contratos_mvno.dtypes

ACORDO_TIPO                   object
PROTOCOLO_DATA                object
PROCESSO_ANATEL               object
PRESTADORA_ORIGEM             object
PRESTADORA_ORIGEM_CNPJ         int64
CREDENCIADA                   object
CREDENCIADA_CNPJ               int64
VIGENCIA_DATA_FIM             object
PROCESSO_DESCREDENCIAMENTO    object
OBSERVACAO                    object
DESPACHO_DECISORIO_SEI         int64
DESPACHO_DECISORIO_DATA       object
CONCLUSAO_DATA                object
dtype: object

In [40]:
df_contratos_mvno.isna().all()

ACORDO_TIPO                   False
PROTOCOLO_DATA                False
PROCESSO_ANATEL               False
PRESTADORA_ORIGEM             False
PRESTADORA_ORIGEM_CNPJ        False
CREDENCIADA                   False
CREDENCIADA_CNPJ              False
VIGENCIA_DATA_FIM             False
PROCESSO_DESCREDENCIAMENTO    False
OBSERVACAO                    False
DESPACHO_DECISORIO_SEI        False
DESPACHO_DECISORIO_DATA       False
CONCLUSAO_DATA                False
dtype: bool

In [41]:
df_contratos_mvno.duplicated().all()

np.False_

In [42]:
df_contratos_mvno.columns

Index(['ACORDO_TIPO', 'PROTOCOLO_DATA', 'PROCESSO_ANATEL', 'PRESTADORA_ORIGEM',
       'PRESTADORA_ORIGEM_CNPJ', 'CREDENCIADA', 'CREDENCIADA_CNPJ',
       'VIGENCIA_DATA_FIM', 'PROCESSO_DESCREDENCIAMENTO', 'OBSERVACAO',
       'DESPACHO_DECISORIO_SEI', 'DESPACHO_DECISORIO_DATA', 'CONCLUSAO_DATA'],
      dtype='object')

In [43]:
df_contratos_mvno[['PROTOCOLO_DATA','DESPACHO_DECISORIO_DATA', 'CONCLUSAO_DATA']]

,PROTOCOLO_DATA,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA
0,07/01/2015,30/01/2015,30/01/2015
1,14/03/2016,20/07/2016,20/07/2016
2,23/03/2016,20/07/2016,20/07/2016
3,23/03/2016,24/08/2016,24/08/2016
4,24/06/2016,17/11/2016,17/11/2016
...,...,...,...
336,21/05/2025,29/05/2025,02/06/2025
337,21/05/2025,29/05/2025,02/06/2025
338,23/05/2025,12/06/2025,16/06/2025
339,29/05/2025,12/06/2025,16/06/2025


In [44]:
df_contratos_mvno.VIGENCIA_DATA_FIM.unique()

array(['07/04/2020', 'VIGENTE', '21/07/2017', '25/01/2019', '08/11/2018',
       '16/08/2021', '18/06/2021', '26/03/2021', '22/03/2023',
       '21/01/2021', '02/10/2024', '06/05/2025', '26/01/2023',
       '22/06/2022', '18/01/2021', '18/01/2022', '01/05/2025',
       '05/03/2024', '03/12/2021', '23/11/2021', '28/01/2022',
       '05/01/2022', '13/12/2021', '07/05/2021', '04/02/2022',
       '21/03/2023', '11/01/2022', '27/12/2022', '10/03/2021',
       '23/01/2023', '18/01/2023', '06/04/2022', '14/03/2022',
       '26/05/2022', '08/04/2022', '19/03/2025', '01/06/2021',
       '04/06/2025', '10/01/2023', '03/01/2022', '09/11/2022',
       '20/12/2023', '08/01/2025', '18/02/2025'], dtype=object)

In [45]:
print(df_contratos_mvno.PRESTADORA_ORIGEM.isna().value_counts())
print(df_contratos_mvno.CREDENCIADA.isna().value_counts())

PRESTADORA_ORIGEM
False    341
Name: count, dtype: int64
CREDENCIADA
False    341
Name: count, dtype: int64


### **Ran Sharing** 

In [48]:
# contratos ran_sharing
df_contratos_ran_sharing = pd.read_csv(
    r'Dados - Contratos/contratos_ran_sharing.csv',
    sep=';'
)
print(df_contratos_ran_sharing.shape)
df_contratos_ran_sharing.head(3)

(17, 20)


,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PROCESSO_ORIGEM,PRESTADORA_1_CNPJ,PRESTADORA_1,PRESTADORA_2_CNPJ,PRESTADORA_2,PRESTADORA_3_CNPJ,PRESTADORA_3,PRESTADORA_4_CNPJ,PRESTADORA_4,ACORDO_TECNOLOGIA,INFORME,INFORME_SEI,INFORME_DATA,ACORDAO,ACORDAO_SEI,ACORDAO_DATA,OBSERVACAO
0,53500.031688/2012-47,14/11/2012,CONTRATO,NaN,4206050000180,TIM CELULAR S.A.,5423963000111,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,NaN,NaN,NaN,NaN,MORAN,Informe nº 01/2013/PVCPR/PVCP/SPV,2754301,21/01/2013,Despacho nº 2719/2013-CD,2754301,21/01/2013,Processo iniciado no SICAP.\r
1,53500.001341/2014-31,09/01/2014,CONTRATO,NaN,66970229000167,CLARO NXT TELECOMUNICACOES LTDA,2558157000162,Telefonica Brasil S.a.,NaN,NaN,NaN,NaN,MOCN,Informe nº 111/2014-PRRE/CPRP/SPR/SCP,2841116,18/02/2014,Acórdão nº 133/2014,2841116,18/02/2014,Processo iniciado no SICAP.\r
2,53500.017260/2015-34,29/07/2015,CONTRATO,NaN,4206050000180,TIM CELULAR S.A.,2558157000162,Telefonica Brasil S.a.,5.423963e+12,OI MÓVEL S.A. - EM RECUPERAÇÃO JUDICIAL,NaN,NaN,MOCN,Informe nº 288/2015-PRRE/ORLE/ORER/CPRP/SPR/SO...,508651,08/10/2015,Acórdão nº 555/2015,508651,08/10/2015,Processo iniciado no SICAP.\r


In [49]:
df_contratos_ran_sharing['ACORDAO'].unique()

array(['Despacho nº 2719/2013-CD', 'Acórdão nº 133/2014',
       'Acórdão nº 555/2015', 'Acórdão nº 199/2014', 'Acórdão nº 22/2015',
       'Acórdão nº 194/2016', 'Acórdão nº 281/2016',
       'Acórdão nº 409/2018', 'Acórdão nº 221/2019', 'Acórdão nº 50/2020',
       'Acórdão nº 209/2020', 'Acórdão nº 286/2021',
       'Acórdão 126 (11993563)'], dtype=object)

In [50]:
df_contratos_ran_sharing.dtypes

PROCESSO_ANATEL       object
PROTOCOLO_DATA        object
ACORDO_TIPO           object
PROCESSO_ORIGEM       object
PRESTADORA_1_CNPJ      int64
PRESTADORA_1          object
PRESTADORA_2_CNPJ      int64
PRESTADORA_2          object
PRESTADORA_3_CNPJ    float64
PRESTADORA_3          object
PRESTADORA_4_CNPJ    float64
PRESTADORA_4         float64
ACORDO_TECNOLOGIA     object
INFORME               object
INFORME_SEI            int64
INFORME_DATA          object
ACORDAO               object
ACORDAO_SEI            int64
ACORDAO_DATA          object
OBSERVACAO            object
dtype: object

In [51]:
df_contratos_ran_sharing.isna().all()

PROCESSO_ANATEL      False
PROTOCOLO_DATA       False
ACORDO_TIPO          False
PROCESSO_ORIGEM      False
PRESTADORA_1_CNPJ    False
PRESTADORA_1         False
PRESTADORA_2_CNPJ    False
PRESTADORA_2         False
PRESTADORA_3_CNPJ    False
PRESTADORA_3         False
PRESTADORA_4_CNPJ     True
PRESTADORA_4          True
ACORDO_TECNOLOGIA    False
INFORME              False
INFORME_SEI          False
INFORME_DATA         False
ACORDAO              False
ACORDAO_SEI          False
ACORDAO_DATA         False
OBSERVACAO           False
dtype: bool

In [52]:
df_contratos_ran_sharing.duplicated()

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15     True
16    False
dtype: bool

In [53]:
df_contratos_ran_sharing[['PRESTADORA_1']]

,PRESTADORA_1
0,TIM CELULAR S.A.
1,CLARO NXT TELECOMUNICACOES LTDA
2,TIM CELULAR S.A.
3,CLARO S.A.
4,CLARO S.A.
5,CLARO S.A.
6,CLARO S.A.
7,CLARO S.A.
8,CLARO NXT TELECOMUNICACOES LTDA
9,TIM CELULAR S.A.


In [54]:
print(df_contratos_ran_sharing['INFORME_DATA'].unique())
print(df_contratos_ran_sharing['ACORDAO_DATA'].unique())
print(df_contratos_ran_sharing['ACORDAO_SEI'].unique())

['21/01/2013' '18/02/2014' '08/10/2015' '03/04/2014' '05/12/2014'
 '05/04/2016' '28/06/2016' '21/06/2018' '22/03/2019' '07/02/2020'
 '17/04/2020' '09/04/2021' '09/05/2023']
['21/01/2013' '18/02/2014' '08/10/2015' '03/04/2014' '05/12/2014'
 '05/04/2016' '28/06/2016' '21/06/2018' '22/03/2019' '07/02/2020'
 '17/04/2020' '09/04/2021' '09/05/2023']
[ 2754301  2841116   508651   392130   392142   526288   720364  2991702
  4101941  5301867  5503736  7295234 11993563]


In [55]:
df_contratos_ran_sharing.ACORDO_TECNOLOGIA.value_counts()

ACORDO_TECNOLOGIA
MOCN         13
GWCN          2
MORAN         1
GWCN/MOCN     1
Name: count, dtype: int64

In [56]:
df_contratos_ran_sharing.PRESTADORA_4.isna().value_counts()

PRESTADORA_4
True    17
Name: count, dtype: int64

In [57]:
df_contratos_ran_sharing.PRESTADORA_4.isna().value_counts()

PRESTADORA_4
True    17
Name: count, dtype: int64

### **Credenciadas Vigentes** 

In [58]:
# empresas credenciadas vigentes
df_empresas_cred = pd.read_csv(
    r'Dados - Contratos/empresas_credenciadas_vigentes.csv',
    sep=';'
)

print(df_empresas_cred.shape)
df_empresas_cred.head(10)

(196, 2)


,PROCESSO_ANATEL,CREDENCIADAS_VIGENTES
0,53504.001925/2016-39,BT Brasil Serviços de Telecomunicações Ltda. (...
1,53500.015283/2016-95,Empresa Brasileira de Correios e Telégrafos (C...
2,53504.000883/2017-08,Eseye do Brasil Tecnologia da Informação Ltda
3,53500.060891/2017-35,DEUTSCHE TELEKOM GLOBAL BUSINESS SOLUTIONS BRA...
4,53504.008572/2017-89,Brisanet Serviços de Telecomunicações Ltda
5,53500.067529/2017-95,GRÊMIO OSASCO AUDAX ESPORTE CLUBE
6,53500.071195/2017-54,Level 3 Comunicações do Brasil LTDA
7,53504.016428/2017-16,Fonelight Telecomunicações S/A
8,53504.019121/2017-77,Fama Serviços Tecnologia da Informação Ltda
9,53504.007684/2018-01,Algar Telecom S.A.


In [59]:
df_empresas_cred.dtypes

PROCESSO_ANATEL          object
CREDENCIADAS_VIGENTES    object
dtype: object

## **EDA Multiplos Datasets** 

In [61]:
df_process = pd.concat([
    df_contratos_int[['PROCESSO_ANATEL']].assign(source='Int'),
    df_contratos_comp[['PROCESSO_ANATEL']].assign(source='Comp'),
    df_contratos_mvno[['PROCESSO_ANATEL']].assign(source='MVNO'),
    df_contratos_ran_sharing[['PROCESSO_ANATEL']].assign(source='RANSh'),
]).drop_duplicates()
df_process = df_process.groupby(['PROCESSO_ANATEL'])[['source']].nunique().reset_index()
df_process[df_process.source > 1]

,PROCESSO_ANATEL,source


In [62]:
def counting_df_processo_anatel(df): 
    print(
        df['PROCESSO_ANATEL']
            .value_counts()
            .reset_index()
            .assign(repeat= lambda x: x['count'] > 1)
            .repeat.value_counts()
    )


In [63]:
df_contratos_int[['PROCESSO_ANATEL']].assign(source='Int'),
df_contratos_comp[['PROCESSO_ANATEL']].assign(source='Comp'),
df_contratos_mvno[['PROCESSO_ANATEL']].assign(source='MVNO'),
df_contratos_ran_sharing[['PROCESSO_ANATEL']].assign(source='RANSh'),

(         PROCESSO_ANATEL source
 0   53500.031688/2012-47  RANSh
 1   53500.001341/2014-31  RANSh
 2   53500.017260/2015-34  RANSh
 3   53500.001089/2014-61  RANSh
 4   53500.020308/2014-19  RANSh
 5   53500.024608/2014-69  RANSh
 6   53500.002241/2016-94  RANSh
 7   53500.002243/2016-83  RANSh
 8   53500.010657/2016-86  RANSh
 9   53500.011812/2018-43  RANSh
 10  53500.043330/2018-52  RANSh
 11  53500.055686/2019-10  RANSh
 12  53500.000608/2020-11  RANSh
 13  53500.001164/2021-12  RANSh
 14  53500.001164/2021-12  RANSh
 15  53500.001164/2021-12  RANSh
 16  53500.315319/2022-86  RANSh,)

In [64]:
counting_df_processo_anatel(df_contratos_int)
counting_df_processo_anatel(df_contratos_comp)
counting_df_processo_anatel(df_contratos_mvno)
counting_df_processo_anatel(df_contratos_ran_sharing)


repeat
True     1261
False     474
Name: count, dtype: int64
repeat
False    2304
True      779
Name: count, dtype: int64
repeat
False    199
True      46
Name: count, dtype: int64
repeat
False    14
True      1
Name: count, dtype: int64


In [65]:
repeated_val = df_contratos_int.PROCESSO_ANATEL.value_counts().index[0]
df_contratos_int[df_contratos_int.PROCESSO_ANATEL == repeated_val]

,PROCESSO_ANATEL,PROTOCOLO_DATA,ACORDO_TIPO,PRESTADORA_1_CNPJ,PRESTADORA_1,SERVICO_1,MODALIDADE_STFC_1,PRESTADORA_2_CNPJ,PRESTADORA_2,SERVICO_2,MODALIDADE_STFC_2,DESPACHO_DECISORIO_SEI,DESPACHO_DECISORIO_DATA,CONCLUSAO_DATA,OBSERVACAO
11212,53500.000155/2024-57,02/01/2024,CONTRATO,2421421000111,TIM S A,10,NaN,25070521000146,R2 Dados Ltda,171,LDI,11492410,09/02/2024,09/02/2024,Não identificado
11213,53500.000155/2024-57,02/01/2024,CONTRATO,2421421000111,TIM S A,10,NaN,25070521000146,R2 Dados Ltda,171,LDN,11492410,09/02/2024,09/02/2024,Não identificado
11214,53500.000155/2024-57,02/01/2024,CONTRATO,2421421000111,TIM S A,10,NaN,25070521000146,R2 Dados Ltda,171,LOCAL,11492410,09/02/2024,09/02/2024,Não identificado
11215,53500.000155/2024-57,02/01/2024,CONTRATO,2421421000111,TIM S A,171,LDI,25070521000146,R2 Dados Ltda,171,LDI,11492410,09/02/2024,09/02/2024,Não identificado
11216,53500.000155/2024-57,02/01/2024,CONTRATO,2421421000111,TIM S A,171,LDI,25070521000146,R2 Dados Ltda,171,LDN,11492410,09/02/2024,09/02/2024,Não identificado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12209,53500.000155/2024-57,12/12/2024,CONTRATO,2421421000111,TIM S A,171,LDN,9399126000155,SERRA GERAL INTERNET LTDA,171,LDN,13145084,20/01/2025,20/01/2025,Homologado por ORPA/OPI.
12210,53500.000155/2024-57,12/12/2024,CONTRATO,2421421000111,TIM S A,171,LDN,9399126000155,SERRA GERAL INTERNET LTDA,171,LOCAL,13145084,20/01/2025,20/01/2025,Homologado por ORPA/OPI.
12211,53500.000155/2024-57,12/12/2024,CONTRATO,2421421000111,TIM S A,171,LOCAL,9399126000155,SERRA GERAL INTERNET LTDA,171,LDI,13145084,20/01/2025,20/01/2025,Homologado por ORPA/OPI.
12212,53500.000155/2024-57,12/12/2024,CONTRATO,2421421000111,TIM S A,171,LOCAL,9399126000155,SERRA GERAL INTERNET LTDA,171,LDN,13145084,20/01/2025,20/01/2025,Homologado por ORPA/OPI.


In [66]:
df_contratos_int[df_contratos_int.PROCESSO_ANATEL == repeated_val].to_clipboard()